<a href="https://colab.research.google.com/github/Frestka/Mining_Premier_League/blob/main/notebooks/Full_Pipeline_Trigger.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    import os, sys
    from pathlib import Path

    REPO_NAME = "Mining_Premier_League"
    REPO_URL  = "https://github.com/Frestka/Mining_Premier_League.git"

    # Torna sempre a /content — evita la matrioska se la cella viene rieseguita
    os.chdir("/content")

    # Clona solo se non esiste già
    if not os.path.exists(REPO_NAME):
        !git clone -q {REPO_URL}
        print("Repository clonato.")

    # Entra nella cartella del progetto
    os.chdir(REPO_NAME)
    sys.path.insert(0, os.path.abspath("."))

    # Aggiorna solo src/ — niente conflitti sui notebook
    !git fetch -q origin
    !git checkout -q origin/main -- src/

    # Installa dipendenze
    !pip install -q -r requirements.txt

    # Crea cartelle data/ e data/processed/ se non esistono
    Path("data/processed").mkdir(parents=True, exist_ok=True)

    # ── Download tutti i file da Google Drive con gdown ──────────────────
    # ISTRUZIONE: sostituisci i placeholder con gli ID reali dei file Drive.
    # Per ottenere l'ID: apri il file su Drive → Condividi → Copia link.
    # L'ID è la stringa tra /d/ e /view del link.
    import gdown

    FILES = {
        "data/events_raw.csv":                  "ID_DRIVE_CSV",    # ~307 MB
        "data/dataset.parquet":                 "ID_DRIVE_BRONZE", # ~37 MB
        "data/processed/dataset_clean.parquet": "ID_DRIVE_SILVER", # ~16 MB
        "data/processed/features.parquet":      "ID_DRIVE_GOLD",   # ~117 KB
    }

    for path, file_id in FILES.items():
        if not os.path.exists(path):
            print(f"Download {path}...")
            gdown.download(id=file_id, output=path, quiet=False, fuzzy=True)
        else:
            print(f"già presente, skip → {path}")

    print(f"\nSetup completato. Cartella: {os.getcwd()}")

else:
    # In locale: risale le cartelle fino a trovare la root (quella con src/)
    import os, sys
    from pathlib import Path
    root = Path.cwd()
    while not (root / "src").exists() and root.parent != root:
        root = root.parent
    if str(root) not in sys.path:
        sys.path.insert(0, str(root))
    os.chdir(root)
    print(f"Ambiente locale. Root progetto: {root}")


#  Full Pipeline Trigger

Questo notebook esegue l'intera pipeline di Data Science in modo pulito ed efficiente.


In [ ]:
import subprocess
import sys
import logging
import os
from pathlib import Path

# Configurazione del logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Assicuriamoci di far partire i comandi dalla cartella radice (root) del progetto
if Path.cwd().name == 'notebooks':
    os.chdir('..')

logger.info(f"Directory di lavoro attuale: {Path.cwd()}")
logger.info(" Avvio della Full Pipeline (Modalità 'In-Place')...")

# Lista dei notebook da aggiornare
notebooks = [
    "notebooks/Data_Cleaning_and_Understanding.ipynb",
    "notebooks/Feature_extraction_EDA.ipynb"
]

# Comando per eseguire in background e salvare i risultati DENTRO i notebook originali
jupyter_cmd = [sys.executable, "-m", "jupyter", "nbconvert", "--to", "notebook", "--execute", "--inplace"]

for nb in notebooks:
    logger.info(f"Esecuzione in corso: {nb} (attendi...)")
    subprocess.run(jupyter_cmd + [nb], check=True)
    logger.info(f" File {nb} aggiornato con i nuovi output!")

logger.info(" Full Pipeline terminata con successo! Dobbiamo aprire i singoli notebook per vedere i risultati.")